In [27]:
import pandas as pd
import re

In [28]:
df = pd.read_csv("raw_data.csv")
df.head()

,Place ID,Nama Tempat,Alamat,Jenis Bisnis,Rating,Jumlah Review,Jam Operasional,Harga Start,Harga End,Latitude,Longitude,Telepon,Website,Keyword
0,ChIJeXbyGyGPcC4Rg7-5wOCeOeA,Bakmi Tantjap Tembalang,"Area NJ Guest House, Jl. Jatimulyo No.9 Lt. 2,...",Restoran,4.8,360,Senin: 10.00–23.30 | Selasa: 10.00–23.30 | Rab...,25000.0,50000.0,-7.056966,110.433502,0851-3858-4778,NaN,bakmi
1,ChIJ5WOtWQCPcC4RedscHY-t5fw,Bakmi Serangkai Tembalang,"Jl. Mulawarman Raya No.11, Pedalangan, Kec. Ba...",Restoran Mie,4.4,81,Senin: 12.00–01.00 | Selasa: 12.00–01.00 | Rab...,25000.0,50000.0,-7.068295,110.433887,0822-2527-1048,NaN,bakmi
2,ChIJTYgZCgCNcC4RmHHSNBKPfYs,Bakmi Polsek Nasihuy Cabang tembalang,"WCWJ+8HX, Jl. Prof. Soedarto, Pedalangan, Kec....",Restoran Indonesia,5.0,3,Senin: Buka 24 jam | Selasa: Buka 24 jam | Rab...,NaN,NaN,-7.054133,110.431475,NaN,NaN,bakmi
3,ChIJhaPc7XaPcC4RjRquGAb9kQE,Bakmi Surabaya Cak Dul 2,"Jl. Mulawarman Selatan Raya No.18, Kramas, Kec...",Restoran Mie,4.0,495,Senin: 07.30–23.30 | Selasa: 07.30–23.30 | Rab...,1.0,25000.0,-7.068204,110.437274,0821-3400-6730,NaN,bakmi
4,ChIJtXUqbImPcC4RdXRvTOFnNlY,BAKSU (Bakmie & Katsu),"Jl. Timoho Bar. III No.38, Bulusan, Kec. Temba...",Restoran Mie,4.3,144,Senin: 09.00–21.00 | Selasa: 09.00–21.00 | Rab...,1.0,25000.0,-7.057076,110.444335,0895-3605-97975,NaN,bakmi


In [29]:
df.shape

(87, 14)

In [43]:
df.dtypes

Place ID            object
Nama Tempat         object
Alamat              object
Jenis Bisnis        object
Rating             float64
Jumlah Review        int64
Jam Operasional     object
Harga Start        float64
Harga End          float64
Latitude           float64
Longitude          float64
Telepon             object
Website             object
Keyword             object
Jalan               object
Kelurahan           object
Kecamatan           object
Kota                object
Provinsi            object
dtype: object

In [30]:
df['Harga Start'].value_counts()

Harga Start
1.0        41
25000.0    20
50000.0     3
75000.0     1
Name: count, dtype: int64

In [41]:
df.duplicated().any()

np.False_

In [31]:
df.isna().sum()

Place ID            0
Nama Tempat         0
Alamat              0
Jenis Bisnis        0
Rating              0
Jumlah Review       0
Jam Operasional     3
Harga Start        22
Harga End          22
Latitude            0
Longitude           0
Telepon            27
Website            77
Keyword             0
dtype: int64

In [32]:
df['Alamat'].unique()

array(['Area NJ Guest House, Jl. Jatimulyo No.9 Lt. 2, Pedalangan, Kec. Banyumanik, Kota Semarang, Jawa Tengah 50268, Indonesia',
       'Jl. Mulawarman Raya No.11, Pedalangan, Kec. Banyumanik, Kota Semarang, Jawa Tengah 50264, Indonesia',
       'WCWJ+8HX, Jl. Prof. Soedarto, Pedalangan, Kec. Banyumanik, Kota Semarang, Jawa Tengah 50268, Indonesia',
       'Jl. Mulawarman Selatan Raya No.18, Kramas, Kec. Tembalang, Kota Semarang, Jawa Tengah 50278, Indonesia',
       'Jl. Timoho Bar. III No.38, Bulusan, Kec. Tembalang, Kota Semarang, Jawa Tengah 50277, Indonesia',
       'Jl. Banjarsari Raya Tembalang No.44, Tembalang, Kec. Tembalang, Kota Semarang, Jawa Tengah 50275, Indonesia',
       'Jl. Sambiroto, Kedungmundu, Kec. Tembalang, Kota Semarang, Jawa Tengah 50276, Indonesia',
       'WCVP+V7G, Jl. Prof. Soedarto, Tembalang, Kec. Tembalang, Kota Semarang, Jawa Tengah 50275, Indonesia',
       'XF4F+PXX, Sendangmulyo, Kec. Tembalang, Kota Semarang, Jawa Tengah 50272, Indonesia',
       

In [33]:
def parse_alamat(alamat):
    if pd.isna(alamat):
        return pd.Series({
            "Jalan": None,
            "Kelurahan": None,
            "Kecamatan": None,
            "Kota": None,
            "Provinsi": None,
        })

    parts = [p.strip() for p in alamat.split(",")]

    # buang Indonesia
    if parts and parts[-1].lower() == "indonesia":
        parts.pop()

    provinsi = None
    kota = None
    kecamatan = None
    kelurahan = None
    jalan = None

    # Provinsi + kode pos
    # contoh: "Jawa Tengah 50196"
    if parts:
        provinsi_raw = parts.pop()
        provinsi = re.sub(r"\s+\d{5}$", "", provinsi_raw).strip()

    # Kota
    # contoh: "Kota Semarang"
    if parts and parts[-1].lower().startswith("kota "):
        kota = re.sub(
            r"^Kota\s+",
            "",
            parts.pop(),
            flags=re.IGNORECASE
        ).strip()

    # Kecamatan
    # contoh: "Kec. Tembalang"
    if parts and re.match(r"^Kec\.?\s+", parts[-1], re.IGNORECASE):
        kecamatan = re.sub(
            r"^Kec\.?\s+",
            "",
            parts.pop(),
            flags=re.IGNORECASE
        ).strip()

    # Segmen tepat sebelum kecamatan = kelurahan
    if parts:
        kelurahan = parts.pop().strip()

    # Semua sisanya dianggap Jalan / detail lokasi
    if parts:
        jalan = ", ".join(parts)

    return pd.Series({
        "Jalan": jalan,
        "Kelurahan": kelurahan,
        "Kecamatan": kecamatan,
        "Kota": kota,
        "Provinsi": provinsi,
    })

In [34]:
df[
    ["Jalan", "Kelurahan", "Kecamatan", "Kota", "Provinsi"]
] = df["Alamat"].apply(parse_alamat)

In [35]:
df.head()

,Place ID,Nama Tempat,Alamat,Jenis Bisnis,Rating,Jumlah Review,Jam Operasional,Harga Start,Harga End,Latitude,Longitude,Telepon,Website,Keyword,Jalan,Kelurahan,Kecamatan,Kota,Provinsi
0,ChIJeXbyGyGPcC4Rg7-5wOCeOeA,Bakmi Tantjap Tembalang,"Area NJ Guest House, Jl. Jatimulyo No.9 Lt. 2,...",Restoran,4.8,360,Senin: 10.00–23.30 | Selasa: 10.00–23.30 | Rab...,25000.0,50000.0,-7.056966,110.433502,0851-3858-4778,NaN,bakmi,"Area NJ Guest House, Jl. Jatimulyo No.9 Lt. 2",Pedalangan,Banyumanik,Semarang,Jawa Tengah
1,ChIJ5WOtWQCPcC4RedscHY-t5fw,Bakmi Serangkai Tembalang,"Jl. Mulawarman Raya No.11, Pedalangan, Kec. Ba...",Restoran Mie,4.4,81,Senin: 12.00–01.00 | Selasa: 12.00–01.00 | Rab...,25000.0,50000.0,-7.068295,110.433887,0822-2527-1048,NaN,bakmi,Jl. Mulawarman Raya No.11,Pedalangan,Banyumanik,Semarang,Jawa Tengah
2,ChIJTYgZCgCNcC4RmHHSNBKPfYs,Bakmi Polsek Nasihuy Cabang tembalang,"WCWJ+8HX, Jl. Prof. Soedarto, Pedalangan, Kec....",Restoran Indonesia,5.0,3,Senin: Buka 24 jam | Selasa: Buka 24 jam | Rab...,NaN,NaN,-7.054133,110.431475,NaN,NaN,bakmi,"WCWJ+8HX, Jl. Prof. Soedarto",Pedalangan,Banyumanik,Semarang,Jawa Tengah
3,ChIJhaPc7XaPcC4RjRquGAb9kQE,Bakmi Surabaya Cak Dul 2,"Jl. Mulawarman Selatan Raya No.18, Kramas, Kec...",Restoran Mie,4.0,495,Senin: 07.30–23.30 | Selasa: 07.30–23.30 | Rab...,1.0,25000.0,-7.068204,110.437274,0821-3400-6730,NaN,bakmi,Jl. Mulawarman Selatan Raya No.18,Kramas,Tembalang,Semarang,Jawa Tengah
4,ChIJtXUqbImPcC4RdXRvTOFnNlY,BAKSU (Bakmie & Katsu),"Jl. Timoho Bar. III No.38, Bulusan, Kec. Temba...",Restoran Mie,4.3,144,Senin: 09.00–21.00 | Selasa: 09.00–21.00 | Rab...,1.0,25000.0,-7.057076,110.444335,0895-3605-97975,NaN,bakmi,Jl. Timoho Bar. III No.38,Bulusan,Tembalang,Semarang,Jawa Tengah


In [36]:
df = df[df['Kecamatan'] == 'Tembalang']

In [37]:
df.head()

,Place ID,Nama Tempat,Alamat,Jenis Bisnis,Rating,Jumlah Review,Jam Operasional,Harga Start,Harga End,Latitude,Longitude,Telepon,Website,Keyword,Jalan,Kelurahan,Kecamatan,Kota,Provinsi
3,ChIJhaPc7XaPcC4RjRquGAb9kQE,Bakmi Surabaya Cak Dul 2,"Jl. Mulawarman Selatan Raya No.18, Kramas, Kec...",Restoran Mie,4.0,495,Senin: 07.30–23.30 | Selasa: 07.30–23.30 | Rab...,1.0,25000.0,-7.068204,110.437274,0821-3400-6730,NaN,bakmi,Jl. Mulawarman Selatan Raya No.18,Kramas,Tembalang,Semarang,Jawa Tengah
4,ChIJtXUqbImPcC4RdXRvTOFnNlY,BAKSU (Bakmie & Katsu),"Jl. Timoho Bar. III No.38, Bulusan, Kec. Temba...",Restoran Mie,4.3,144,Senin: 09.00–21.00 | Selasa: 09.00–21.00 | Rab...,1.0,25000.0,-7.057076,110.444335,0895-3605-97975,NaN,bakmi,Jl. Timoho Bar. III No.38,Bulusan,Tembalang,Semarang,Jawa Tengah
5,ChIJJatTaFuPcC4RgzWs9JqpT4w,"Pondok Bakmie Surabaya ""Cak Dul""","Jl. Banjarsari Raya Tembalang No.44, Tembalang...",Restoran Mie,3.4,64,Senin: 07.00–23.30 | Selasa: 07.00–23.30 | Rab...,1.0,25000.0,-7.059733,110.440100,0813-2803-2121,NaN,bakmi,Jl. Banjarsari Raya Tembalang No.44,Tembalang,Tembalang,Semarang,Jawa Tengah
6,ChIJ6c-rHNeNcC4RxFrb1LRitnY,Bakmi Juned,"Jl. Sambiroto, Kedungmundu, Kec. Tembalang, Ko...",Restoran Mie,4.7,158,Senin: Tutup | Selasa: 17.00–00.00 | Rabu: 17....,1.0,25000.0,-7.024015,110.458790,NaN,NaN,bakmi,Jl. Sambiroto,Kedungmundu,Tembalang,Semarang,Jawa Tengah
7,ChIJecbeHbSNcC4R8qA_dqzD6_0,Bakmi Jowo Pak Pardi,"WCVP+V7G, Jl. Prof. Soedarto, Tembalang, Kec. ...",Restoran Mie,4.5,73,Senin: 17.30–23.00 | Selasa: 17.30–23.00 | Rab...,1.0,25000.0,-7.055324,110.435704,NaN,NaN,bakmi,"WCVP+V7G, Jl. Prof. Soedarto",Tembalang,Tembalang,Semarang,Jawa Tengah


In [38]:
df.shape

(63, 19)

In [42]:
df[df["Harga Start"].isna()]

,Place ID,Nama Tempat,Alamat,Jenis Bisnis,Rating,Jumlah Review,Jam Operasional,Harga Start,Harga End,Latitude,Longitude,Telepon,Website,Keyword,Jalan,Kelurahan,Kecamatan,Kota,Provinsi
9,ChIJQx9PIwCNcC4R_y1_VUp6Slw,Bakmie Polsek Nasihuy cabang tembalang,"Jl. Adi Patiunus No.16-2, Tembalang, Kec. Temb...",Restoran Indonesia,5.0,1,Senin: Buka 24 jam | Selasa: Buka 24 jam | Rab...,NaN,NaN,-7.055036,110.434693,NaN,NaN,bakmi,Jl. Adi Patiunus No.16-2,Tembalang,Tembalang,Semarang,Jawa Tengah
14,ChIJtd7IDgCPcC4RDenSnPjmk18,PONDOK Bakmi Surabaya Cak DUL,"WFV7+478, Jl. Profesor Suharso, Meteseh, Kec. ...",Warung Makanan Kecil,3.9,14,Senin: 16.00–23.00 | Selasa: 16.00–23.00 | Rab...,NaN,NaN,-7.057205,110.463189,0813-2515-7257,NaN,bakmi,"WFV7+478, Jl. Profesor Suharso",Meteseh,Tembalang,Semarang,Jawa Tengah
15,ChIJrSiv_0uMcC4R7u6wqeW5U2k,Bakmi Surabaya Cak Baron,"XFC9+768, Jl. Ketileng Raya, Sendangmulyo, Kec...",Restoran Mie,4.6,27,Senin: 18.45–23.45 | Selasa: 18.45–23.45 | Rab...,NaN,NaN,-7.029325,110.468078,NaN,NaN,bakmi,"XFC9+768, Jl. Ketileng Raya",Sendangmulyo,Tembalang,Semarang,Jawa Tengah
21,ChIJoVUk1V2NcC4RLXWsQyrPNgQ,BAKMI JOWO KANG DOEL,"XFH4+HPR, Kedungmundu, Kec. Tembalang, Kota Se...",Restoran Mie,4.9,16,Senin: 15.30–23.30 | Selasa: 15.30–23.30 | Rab...,NaN,NaN,-7.021003,110.456780,0812-2556-0512,NaN,bakmi,XFH4+HPR,Kedungmundu,Tembalang,Semarang,Jawa Tengah
24,ChIJ6fMwdzaMcC4R8Y2y6XRoDio,Bakmi Jowo Mas Agus,"XF69+X82, Jl. Gendong Raya, Mangunharjo, Kec. ...",Restoran Mie,4.6,29,Senin: 17.00–00.00 | Selasa: 17.00–00.00 | Rab...,NaN,NaN,-7.037619,110.468281,0819-0437-2937,NaN,bakmi,"XF69+X82, Jl. Gendong Raya",Mangunharjo,Tembalang,Semarang,Jawa Tengah
27,ChIJJxgFttKNcC4RQ8rAtfeapGY,bakmi surabaya cak yanto,"XF3F+VP9, Sendangmulyo, Kec. Tembalang, Kota S...",Restoran Mie,4.5,15,Senin: 17.00–22.00 | Selasa: 17.00–22.00 | Rab...,NaN,NaN,-7.045329,110.474352,0895-3367-24891,NaN,bakmi,XF3F+VP9,Sendangmulyo,Tembalang,Semarang,Jawa Tengah
29,ChIJrQVc_UaMcC4RTZfwIkFLgi4,Bakmi Jowo Mas Agus,"Jl. Sambiroto Raya No.6, Sambiroto, Kec. Temba...",Restoran Mie,4.5,99,Senin: 17.00–02.00 | Selasa: 17.00–02.00 | Rab...,NaN,NaN,-7.032360,110.458110,0858-0213-9915,NaN,bakmi,Jl. Sambiroto Raya No.6,Sambiroto,Tembalang,Semarang,Jawa Tengah
32,ChIJJxXt44COcC4RctS71K6-h98,Bakmi Surabaya,"Jalan Rowosari Raya, Jl. Talang Jl. Baskoro Ra...",Restoran Mie,3.8,10,Senin: 17.00–00.00 | Selasa: 17.00–00.00 | Rab...,NaN,NaN,-7.058050,110.471033,0813-2669-4979,NaN,bakmi,"Jalan Rowosari Raya, Jl. Talang Jl. Baskoro Raya",Meteseh,Tembalang,Semarang,Jawa Tengah
34,ChIJ3Z0aT0SNcC4R5DYgdoElS6I,"BAKMI SUROBOYO MAS NDUT (YUDI),METESEH","Jl. Raya Sendangmulyo, Meteseh, Kec. Tembalang...",Restoran Mie,4.3,8,Senin: 17.30–23.00 | Selasa: 17.30–23.00 | Rab...,NaN,NaN,-7.052854,110.471467,0878-3106-0032,NaN,bakmi,Jl. Raya Sendangmulyo,Meteseh,Tembalang,Semarang,Jawa Tengah
39,ChIJKQGsi8CNcC4RMUH7EycP9o0,bakmie surabaya mas allfa99,"Jl. Kedungmundu No.169, Sendangguwo, Kec. Temb...",Konsultan,5.0,3,Senin: 17.00–23.00 | Selasa: 16.30–23.30 | Rab...,NaN,NaN,-7.013551,110.451746,0822-2408-0549,http://grabfood.co.id/,bakmi,Jl. Kedungmundu No.169,Sendangguwo,Tembalang,Semarang,Jawa Tengah


In [44]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

print("Columns:")
print(df.columns.tolist())

Columns:
['place_id', 'nama_tempat', 'alamat', 'jenis_bisnis', 'rating', 'jumlah_review', 'jam_operasional', 'harga_start', 'harga_end', 'latitude', 'longitude', 'telepon', 'website', 'keyword', 'jalan', 'kelurahan', 'kecamatan', 'kota', 'provinsi']


In [45]:
print("\nMissing values:")
print(df.isna().sum())

print("\nData types:")
print(df.dtypes)


Missing values:
place_id            0
nama_tempat         0
alamat              0
jenis_bisnis        0
rating              0
jumlah_review       0
jam_operasional     3
harga_start        20
harga_end          20
latitude            0
longitude           0
telepon            20
website            60
keyword             0
jalan               3
kelurahan           0
kecamatan           0
kota                0
provinsi            0
dtype: int64

Data types:
place_id            object
nama_tempat         object
alamat              object
jenis_bisnis        object
rating             float64
jumlah_review        int64
jam_operasional     object
harga_start        float64
harga_end          float64
latitude           float64
longitude          float64
telepon             object
website             object
keyword             object
jalan               object
kelurahan           object
kecamatan           object
kota                object
provinsi            object
dtype: object


In [46]:
df.to_csv(
    "clean_data.csv",
    index=False,
    encoding="utf-8-sig"
)